# 04 - Interpretability: exact Shapley values and permutation importance

Explains the single-split TimeMixer ensemble (10 seeds, config A) on the **test** block.

- **Exact Shapley values** over the 7 input feature groups: all 2^7 = 128 coalitions are
  evaluated; features outside a coalition are replaced by windows from 10 random training
  samples (background).
- **Permutation importance**: increase in RMSE when one feature is shuffled across samples.

**Needs:** outputs of notebooks 01 and 02. Trained weights are cached in `results/models/`.
**Writes:** `results/shap_importance.csv`, `results/models/tm_seed*.pt`.

In [ ]:
import itertools
import os
import sys
from math import factorial
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# The notebooks live in notebooks/, but all paths below are relative to the repository root.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

# TimeMixer reference implementation (git submodule in TimeMixer/)
sys.path.insert(0, "TimeMixer")
from models.TimeMixer import Model  # noqa: E402

## 1. Load the windows and the 10-seed ensemble

In [ ]:
# Standardised windows from 01_data_preparation
d = np.load("data/windows_N64.npz", allow_pickle=True)
Xtr, ytr, Xva, yva, Xte, yte = [d[k] for k in ["Xtr", "ytr", "Xva", "yva", "Xte", "yte"]]
y_mu, y_sd = float(d["y_mu"]), float(d["y_sd"])

# TimeMixer configuration A: d_model 8, d_ff 16
cfg = SimpleNamespace(
    task_name="long_term_forecast",
    seq_len=64,
    label_len=0,
    pred_len=1,
    enc_in=7,
    c_out=7,
    d_model=8,
    d_ff=16,
    e_layers=2,
    dropout=0.2,
    down_sampling_layers=3,
    down_sampling_window=2,
    down_sampling_method="avg",
    channel_independence=0,
    use_norm=0,
    decomp_method="moving_avg",
    moving_avg=5,
    top_k=5,
    embed="timeF",
    freq="d",
    use_future_temporal_feature=0,
    num_class=0,
)

In [ ]:
class Net(nn.Module):
    """TimeMixer core followed by a linear read-out over its 7 output channels."""

    def __init__(self, core):
        super().__init__()
        self.core = core
        self.readout = nn.Linear(7, 1)

    def forward(self, x):
        z = self.core(x, None, None, None)
        return self.readout(z[:, 0, :]).squeeze(-1)


to_t = lambda a: torch.tensor(a, dtype=torch.float32)
Xtr_t, ytr_t, Xva_t, yva_t, Xte_t = map(to_t, [Xtr, ytr, Xva, yva, Xte])

In [ ]:
def train_seed(s):
    """Train one network (seed s) with early stopping on the validation loss."""
    torch.manual_seed(s)
    np.random.seed(s)
    net = Net(Model(cfg))
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    best, best_state, bad = float("inf"), None, 0
    for epoch in range(100):
        net.train()
        perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(perm), 32):
            idx = perm[i : i + 32]
            opt.zero_grad()
            loss_fn(net(Xtr_t[idx]), ytr_t[idx]).backward()
            opt.step()
        net.eval()
        with torch.no_grad():
            v = loss_fn(net(Xva_t), yva_t).item()
        if v < best - 1e-5:
            best, bad = v, 0
            best_state = {k: x.clone() for k, x in net.state_dict().items()}
        else:
            bad += 1
            if bad >= 10:
                break
    net.load_state_dict(best_state)
    return net.eval()

In [1]:
# Load cached weights when available, otherwise train and cache them
os.makedirs("results/models", exist_ok=True)
nets = []
for s in range(10):
    path = Path(f"results/models/tm_seed{s}.pt")
    if path.exists():
        n = Net(Model(cfg))
        n.load_state_dict(torch.load(path))
        n.eval()
    else:
        n = train_seed(s)
        torch.save(n.state_dict(), path)
    nets.append(n)


def ens_predict(X_t):
    """Ensemble forecast in volatility units (undo the scaling and the log)."""
    with torch.no_grad():
        return np.mean([np.exp(n(X_t).numpy() * y_sd + y_mu) for n in nets], axis=0)


val = pd.read_csv("results/val_predictions.csv", parse_dates=["Date"])
test = pd.read_csv("results/test_predictions.csv", parse_dates=["Date"])
yv = val["actual"].values
yt = test["actual"].values
rm = lambda y, p: round(float(np.sqrt(np.mean((y - p) ** 2))), 3)
print("val ensemble RMSE (expect 1.589):", rm(yv, ens_predict(Xva_t)))
print("test ensemble RMSE (expect 1.819):", rm(yt, ens_predict(Xte_t)))
print("saved models:", len(list(Path("results/models").glob("*.pt"))))

val ensemble RMSE (expect 1.589): 1.589
test ensemble RMSE (expect 1.819): 1.819
saved models: 10


## 2. Exact Shapley values (test block)

In [2]:
F = 7
names = ["realized_vol_22d", "park_d", "park_w", "park_m", "IMPL_VOL_1M", "VOL_10D", "VOL_90D"]

rng = np.random.default_rng(0)
K = 10
bg = Xtr_t[rng.choice(len(Xtr_t), K, replace=False)]  # (K, 64, 7)

masks = np.array(list(itertools.product([0, 1], repeat=F)), dtype=np.float32)  # (128, 7)
masks_t = torch.tensor(masks)[:, None, None, :]
size = masks.sum(1).astype(int)
idx = {tuple(m.astype(int)): i for i, m in enumerate(masks)}


def f_ens(X):
    with torch.no_grad():
        return torch.stack([n(X) for n in nets]).mean(0)


def coalition_values(x):  # x: (64, 7)
    mixed = masks_t * x[None, None] + (1 - masks_t) * bg[None]  # (128, K, 64, 7)
    return f_ens(mixed.reshape(-1, 64, F)).reshape(128, K).mean(1).numpy()


def shapley(v):
    phi = np.zeros(F)
    for j in range(F):
        for i, m in enumerate(masks):
            if m[j] == 0:
                s = size[i]
                w = factorial(s) * factorial(F - s - 1) / factorial(F)
                m2 = m.copy()
                m2[j] = 1
                phi[j] += w * (v[idx[tuple(m2.astype(int))]] - v[i])
    return phi


phis, vals = [], []
for i in range(len(Xte_t)):
    v = coalition_values(Xte_t[i])
    vals.append(v)
    phis.append(shapley(v))
phis, vals = np.array(phis), np.array(vals)

# additivity check: contributions must sum to (prediction - baseline)
pred_full = f_ens(Xte_t).numpy()
print("additivity max error:", float(np.abs(phis.sum(1) - (vals[:, -1] - vals[:, 0])).max()))
print("v(all on) matches model output:", float(np.abs(vals[:, -1] - pred_full).max()))

imp = np.abs(phis).mean(0) * y_sd * 100
res_df = pd.DataFrame(
    {
        "feature": names,
        "mean |contribution| (%)": imp.round(2),
        "share (%)": (100 * imp / imp.sum()).round(1),
        "mean signed (%)": (phis.mean(0) * y_sd * 100).round(2),
    }
).sort_values("share (%)", ascending=False)
print(res_df.to_string(index=False))

res_df.to_csv("results/shap_importance.csv", index=False)

additivity max error: 1.0152143659070134e-07
v(all on) matches model output: 3.5762786865234375e-07
         feature  mean |contribution| (%)  share (%)  mean signed (%)
     IMPL_VOL_1M                    16.31       38.7           -16.31
          park_w                    12.13       28.8           -12.13
          park_d                     3.62        8.6             3.57
realized_vol_22d                     3.44        8.2            -1.27
         VOL_90D                     2.65        6.3            -0.30
         VOL_10D                     2.06        4.9            -0.37
          park_m                     1.92        4.6            -1.58


## 3. Permutation importance

Each feature is shuffled across samples (5 repetitions); a larger RMSE increase means the
model relies more on that feature. Run on the validation block and on the test block.

In [3]:
# Permutation importance on the validation block
def perm_rmse(X_t, y, j, reps=5):
    out = []
    for r in range(reps):
        g = torch.Generator().manual_seed(r)
        p = torch.randperm(len(X_t), generator=g)
        X2 = X_t.clone()
        X2[:, :, j] = X_t[p][:, :, j]
        out.append(np.sqrt(np.mean((y - ens_predict(X2)) ** 2)))
    return float(np.mean(out))


base = float(np.sqrt(np.mean((yv - ens_predict(Xva_t)) ** 2)))
print("baseline val RMSE:", round(base, 3))
for j, n in enumerate(names):
    print(f"{n:18s} RMSE when shuffled: {perm_rmse(Xva_t, yv, j):.3f}")

baseline val RMSE: 1.589
realized_vol_22d   RMSE when shuffled: 1.735
park_d             RMSE when shuffled: 1.633
park_w             RMSE when shuffled: 1.849
park_m             RMSE when shuffled: 1.627
IMPL_VOL_1M        RMSE when shuffled: 1.876
VOL_10D            RMSE when shuffled: 1.728
VOL_90D            RMSE when shuffled: 1.590


In [4]:
# Test block
base_t = float(np.sqrt(np.mean((yt - ens_predict(Xte_t)) ** 2)))
print("baseline test RMSE:", round(base_t, 3))
for j, n in enumerate(names):
    print(f"{n:18s} RMSE when shuffled: {perm_rmse(Xte_t, yt, j):.3f}")

baseline test RMSE: 1.819
realized_vol_22d   RMSE when shuffled: 1.889
park_d             RMSE when shuffled: 1.840
park_w             RMSE when shuffled: 1.728
park_m             RMSE when shuffled: 1.790
IMPL_VOL_1M        RMSE when shuffled: 1.747
VOL_10D            RMSE when shuffled: 1.816
VOL_90D            RMSE when shuffled: 1.727
